# LightGBM Text Report Visualizer

Reads a `LightGBM_2.txt` report and generates:
- `classification_report.csv`
- `confusion_matrix.png`
- `f1_scores.png`


In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import confusion_matrix


In [ ]:
def parse_classification_report(lines):
    start = None
    for i, line in enumerate(lines):
        if line.strip() == "DETAILED CLASSIFICATION REPORT":
            start = i
            break
    if start is None:
        raise ValueError("Could not find 'DETAILED CLASSIFICATION REPORT' section.")

    # Find header line
    header_idx = None
    for i in range(start, len(lines)):
        if lines[i].strip().startswith("precision"):
            header_idx = i
            break
    if header_idx is None:
        raise ValueError("Could not find report header line.")

    rows = []
    for line in lines[header_idx + 1:]:
        if not line.strip():
            break
        parts = line.split()
        if len(parts) < 4:
            continue
        label = parts[0]
        precision = float(parts[1])
        recall = float(parts[2])
        f1 = float(parts[3])
        support = int(parts[4]) if len(parts) > 4 else 0
        rows.append((label, precision, recall, f1, support))
    report_df = pd.DataFrame(rows, columns=["label", "precision", "recall", "f1-score", "support"]).set_index("label")
    return report_df


def parse_confusion_matrix(lines):
    start = None
    for i, line in enumerate(lines):
        if line.strip() == "CONFUSION MATRIX":
            start = i
            break
    if start is None:
        raise ValueError("Could not find 'CONFUSION MATRIX' section.")

    header_idx = None
    for i in range(start + 1, len(lines)):
        if lines[i].strip().startswith("True\\Pred"):
            header_idx = i
            break
    if header_idx is None:
        raise ValueError("Could not find confusion matrix header line.")

    header_parts = lines[header_idx].split()
    labels = header_parts[1:]

    matrix = []
    for line in lines[header_idx + 1:]:
        if not line.strip():
            break
        if line.strip().startswith("-"):
            continue
        parts = line.split()
        if len(parts) < 2:
            continue
        row_vals = [int(x) for x in parts[1:] if x.isdigit()]
        if row_vals:
            matrix.append(row_vals)
    cm = np.array(matrix)
    return labels, cm


def plot_confusion(cm, labels, out_path: Path, title: str):
    plt.figure(figsize=(12, 10))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
        cbar=False,
    )
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.close()


In [ ]:
txt_path = Path("LightGBM_2.txt")
out_dir = Path.cwd()

text = txt_path.read_text(encoding="utf-8", errors="ignore")
lines = text.splitlines()

report_df = parse_classification_report(lines)
labels, cm = parse_confusion_matrix(lines)

report_path = out_dir / "classification_report.csv"
report_df.to_csv(report_path, index=True)

cm_path = out_dir / "confusion_matrix.png"
plot_confusion(cm, labels, cm_path, "Confusion Matrix (Test)")

f1_scores = report_df.loc[labels, "f1-score"]
plt.figure(figsize=(12, 4))
sns.barplot(x=f1_scores.index, y=f1_scores.values, color="steelblue")
plt.title("Per-class F1-score (Test)")
plt.ylabel("F1-score")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(out_dir / "f1_scores.png", dpi=150)
plt.close()

print("Saved:")
print(report_path)
print(cm_path)
print(out_dir / "f1_scores.png")
report_df
